In [1]:
import csv
import os
import requests

# 이번에는 디코딩 키(Decoding)를 사용합니다.
DECODING_KEY = "HpJGL7BvOA3rXvZz45iwccKnjBch2u4h7FrglCEn4xEsWj1Pm/ILD9PLQlg4ark72DzBEJxhzFWT9fo1sJ65mw=="
URL = "https://apis.data.go.kr/1543061/abandonmentPublicService_v2/abandonmentPublic_v2"

os.makedirs("data", exist_ok=True)
OUTPUT_FILE = "data/raw_animals_sample.csv"

# requests.get에 params로 디코딩 키를 넘기면 라이브러리가 알아서 인코딩해서 보냅니다.
params = {
    "serviceKey": DECODING_KEY,
    "bgnde": "20250101",
    "endde": "20250131",
    "pageNo": "1",
    "numOfRows": "100",
    "_type": "json",
}

try:
    print("디코딩 키 + params 정석 조합으로 재시도 중...")
    response = requests.get(URL, params=params)

    print(f"응답 상태 코드: {response.status_code}")

    if response.status_code == 200:
        res_data = response.json()
        body = res_data.get("response", {}).get("body", {})
        items = body.get("items", {}).get("item", [])

        if items:
            if isinstance(items, dict):
                items = [items]
            headers = items[0].keys()
            with open(OUTPUT_FILE, "w", newline="", encoding="utf-8-sig") as f:
                writer = csv.DictWriter(f, fieldnames=headers)
                writer.writeheader()
                writer.writerows(items)
            print(f"🎉 성공! {len(items)}건 저장 완료.")
        else:
            print("⚠️ 연결 성공, 내부 데이터 없음:", res_data)
    else:
        print(f"❌ 여전히 실패 (상태 코드: {response.status_code})")

except Exception as e:
    print(f"🚨 에러: {e}")

디코딩 키 + params 정석 조합으로 재시도 중...
응답 상태 코드: 200
🚨 에러: dict contains fields not in fieldnames: 'popfile5', 'popfile3'


In [2]:
import os
import pandas as pd
import requests

# 1. 인증키 및 기본 URL 설정
DECODING_KEY = "HpJGL7BvOA3rXvZz45iwccKnjBch2u4h7FrglCEn4xEsWj1Pm/ILD9PLQlg4ark72DzBEJxhzFWT9fo1sJ65mw=="
URL = "https://apis.data.go.kr/1543061/abandonmentPublicService_v2/abandonmentPublic_v2"

# 2. 저장 경로 설정
os.makedirs("data", exist_ok=True)
OUTPUT_FILE = "data/raw_animals_sample.csv"

# 3. 요청 파라미터 (2025년 1월 데이터 100건)
params = {
    "serviceKey": DECODING_KEY,
    "bgnde": "20250101",
    "endde": "20250131",
    "pageNo": "1",
    "numOfRows": "100",
    "_type": "json",
}

try:
    print("Pandas 기반 안전한 방식으로 오픈 API 호출 중...")
    response = requests.get(URL, params=params)

    if response.status_code == 200:
        res_data = response.json()
        body = res_data.get("response", {}).get("body", {})
        items_wrapper = body.get("items", {})

        if items_wrapper and "item" in items_wrapper:
            items = items_wrapper["item"]
            # 데이터가 1건일 경우 딕셔너리로 오는 현상 방지
            if isinstance(items, dict):
                items = [items]
        else:
            items = []

        if items:
            # ⭐ Pandas DataFrame으로 변환 (들쭉날쭉한 필드를 알아서 표 형태로 정렬해 줌)
            df = pd.DataFrame(items)

            # 엑셀 및 Spark 한글 깨짐 방지를 위해 utf-8-sig로 CSV 저장
            df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
            print(
                f"🎉 대성공! {len(df)}건의 데이터가 유연하게 정렬되어 '{OUTPUT_FILE}'에 저장되었습니다."
            )
            print(f"생성된 컬럼 개수: {len(df.columns)}개")
        else:
            print("⚠️ 연결은 성공했으나 데이터가 없습니다.")
    else:
        print(f"❌ API 호출 실패 (상태 코드: {response.status_code})")

except Exception as e:
    print(f"🚨 에러 발생: {e}")

Pandas 기반 안전한 방식으로 오픈 API 호출 중...
🎉 대성공! 100건의 데이터가 유연하게 정렬되어 'data/raw_animals_sample.csv'에 저장되었습니다.
생성된 컬럼 개수: 35개


In [3]:
import os
import time
import pandas as pd
import requests

# 1. 인증키 및 기본 URL 설정
DECODING_KEY = "HpJGL7BvOA3rXvZz45iwccKnjBch2u4h7FrglCEn4xEsWj1Pm/ILD9PLQlg4ark72DzBEJxhzFWT9fo1sJ65mw=="
URL = "https://apis.data.go.kr/1543061/abandonmentPublicService_v2/abandonmentPublic_v2"

# 2. 과제 요구사항에 맞춘 데이터 저장 경로 생성
os.makedirs("data", exist_ok=True)
OUTPUT_FILE = "data/raw_animals_full.csv"

# 100MB 이상 확보를 위한 타겟 연도 (최근 3개년 수집)
TARGET_YEARS = ["2023", "2024", "2025"]
all_data = []

print("✅ 기본 설정 완료! 다음 셀을 실행하여 수집을 시작하세요.")

✅ 기본 설정 완료! 다음 셀을 실행하여 수집을 시작하세요.


In [4]:
print("======= 🚀 유기동물 대용량 데이터 수집 시작 =======")

for year in TARGET_YEARS:
    print(f"\n📅 {year}년도 데이터 수집 중...")

    # API 서버의 타임아웃을 방지하기 위해 상반기/하반기로 쪼개서 요청
    periods = [
        {"bgnde": f"{year}0101", "endde": f"{year}0630"},
        {"bgnde": f"{year}0701", "endde": f"{year}1231"},
    ]

    for period in periods:
        page = 1
        while True:
            params = {
                "serviceKey": DECODING_KEY,
                "bgnde": period["bgnde"],
                "endde": period["endde"],
                "pageNo": str(page),
                "numOfRows": "1000",  # 한 페이지당 최대 1000건씩 요청
                "_type": "json",
            }

            try:
                response = requests.get(URL, params=params, timeout=30)

                if response.status_code == 200:
                    res_data = response.json()
                    body = res_data.get("response", {}).get("body", {})
                    items_wrapper = body.get("items", {})

                    if items_wrapper and "item" in items_wrapper:
                        items = items_wrapper["item"]
                        if isinstance(items, dict):
                            items = [items]

                        # 데이터프레임으로 변환 후 리스트에 저장
                        df_page = pd.DataFrame(items)
                        all_data.append(df_page)

                        print(
                            f"   [성공] {period['bgnde']}~{period['endde']} | {page}페이지 - {len(df_page)}건 추가됨"
                        )

                        if len(df_page) < 1000:  # 1000건 미만이면 마지막 페이지임
                            break

                        page += 1
                        time.sleep(0.2)  # 서버 차단 방지 디레이
                    else:
                        break
                else:
                    print(
                        f"   [에러] 상태 코드 {response.status_code}. 5초 후 재시도..."
                    )
                    time.sleep(5)
                    continue

            except Exception as e:
                print(f"   [경고] 예외 발생: {e}. 5초 후 재시도...")
                time.sleep(5)
                continue

print("\n======= 🌟 모든 기간 API 호출 완료! =======")

======= 🚀 유기동물 대용량 데이터 수집 시작 =======

📅 2023년도 데이터 수집 중...
   [성공] 20230101~20230630 | 1페이지 - 1000건 추가됨
   [성공] 20230101~20230630 | 2페이지 - 1000건 추가됨
   [성공] 20230101~20230630 | 3페이지 - 1000건 추가됨
   [성공] 20230101~20230630 | 4페이지 - 1000건 추가됨
   [성공] 20230101~20230630 | 5페이지 - 1000건 추가됨
   [성공] 20230101~20230630 | 6페이지 - 1000건 추가됨
   [성공] 20230101~20230630 | 7페이지 - 1000건 추가됨
   [성공] 20230101~20230630 | 8페이지 - 1000건 추가됨
   [성공] 20230101~20230630 | 9페이지 - 1000건 추가됨
   [성공] 20230101~20230630 | 10페이지 - 1000건 추가됨
   [성공] 20230101~20230630 | 11페이지 - 1000건 추가됨
   [성공] 20230101~20230630 | 12페이지 - 1000건 추가됨
   [성공] 20230101~20230630 | 13페이지 - 1000건 추가됨
   [성공] 20230101~20230630 | 14페이지 - 1000건 추가됨
   [성공] 20230101~20230630 | 15페이지 - 1000건 추가됨
   [성공] 20230101~20230630 | 16페이지 - 1000건 추가됨
   [성공] 20230101~20230630 | 17페이지 - 1000건 추가됨
   [성공] 20230101~20230630 | 18페이지 - 1000건 추가됨
   [성공] 20230101~20230630 | 19페이지 - 1000건 추가됨
   [성공] 20230101~20230630 | 20페이지 - 1000건 추가됨
   [성공] 20230101~20230630 | 21

KeyboardInterrupt: 

In [ ]:
if all_data:
    print("======= 💾 데이터 병합 및 정제 시작 =======")
    final_df = pd.concat(all_data, ignore_index=True)

    # 중복 데이터 제거 (고유번호 기준)
    final_df.drop_duplicates(subset=["desertionNo"], inplace=True)

    # UTF-8-SIG 인코딩으로 한글 깨짐 없이 CSV 저장
    final_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    # 최종 용량 계산
    file_size_mb = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)

    print(f"🎉 최종 파일 저장 완료: {OUTPUT_FILE}")
    print(f"📊 총 수집된 유기동물 건수: {len(final_df):,} 건")
    print(f"💾 최종 파일 용량: {file_size_mb:.2f} MB")

    if file_size_mb >= 100:
        print("✅ 교수님 필수 요구사항 (100MB 이상) 충족 완료!")
    else:
        print("⚠️ 용량이 부족합니다. TARGET_YEARS에 연도를 추가하세요.")
else:
    print("❌ 수집된 데이터가 없습니다.")

In [ ]:
# 파일이 잘 저장되었는지 로드해서 상위 5개 행만 출력해보는 셀
df_check = pd.read_csv(OUTPUT_FILE)
print(f"체크된 데이터프레임 형태: {df_check.shape}")
df_check.head()